# RF-DETR Training and Evaluation Pipeline for BCDD-SUST

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q "rfdetr>=1.4.0" supervision roboflow pandas numpy tqdm pillow torch

In [ ]:
# 2. Configuration & Hyperparameters Setup
import os
import torch

# Set seeds for exact reproducibility
SEED = 42
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Dataset and Output Directory Paths
DATASET_ROOT = "Dataset Path"
CHECKPOINT_DIR = "CheckPoint Path"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

NUM_CLASSES = 7  # BCDD-SUST Target Classes
EPOCHS = 50
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
CONFIDENCE_THRESHOLD = 0.25

In [ ]:
# 3. Model Fine-Tuning (Training Phase)
from rfdetr import RFDETRLarge

print("Initializing RF-DETR Large Model...")
model = RFDETRLarge(num_classes=NUM_CLASSES)

print("Starting Model Fine-Tuning...")
model.train(
    dataset_dir=DATASET_ROOT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    output_dir=CHECKPOINT_DIR
)

print(f"Training complete. Best checkpoint saved to: {CHECKPOINT_DIR}/checkpoint_best_ema.pth")

In [ ]:
# 4. Model Loading & Inference Optimization
from rfdetr import RFDETRLarge

best_checkpoint_path = os.path.join(CHECKPOINT_DIR, "checkpoint_best_ema.pth")
model = RFDETRLarge(pretrain_weights=best_checkpoint_path)
model.optimize_for_inference()

In [ ]:
# 5. Test Dataset Preparation
import supervision as sv

test_images_dir = os.path.join(DATASET_ROOT, "test")
test_ann_path = os.path.join(DATASET_ROOT, "test/_annotations.coco.json")

ds = sv.DetectionDataset.from_coco(
    images_directory_path=test_images_dir,
    annotations_path=test_ann_path
)

In [ ]:
# 6. Full Model Evaluation & Metric Computation
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

if hasattr(model, 'model') and hasattr(model.model, 'eval'):
    model.model.eval()

targets = []
predictions = []

print("\nRunning dataset inference...")
with torch.no_grad():
    for path, image, annotations in tqdm(ds):
        with Image.open(path) as img:
            detections = model.predict(img, threshold=0)
        targets.append(annotations)
        predictions.append(detections)

print("\nComputing mAP metrics...")
map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()

print("Computing Confusion Matrix...")
filtered_predictions = [
    pred[pred.confidence > CONFIDENCE_THRESHOLD] for pred in predictions
]

conf_matrix = sv.ConfusionMatrix.from_detections(
    predictions=filtered_predictions,
    targets=targets,
    classes=ds.classes
)

class_names = ds.classes
matrix = conf_matrix.matrix

class_id_to_idx = {cid: idx for idx, cid in enumerate(map_result.matched_classes)}
iou50_idx = np.where(np.isclose(map_result.iou_thresholds, 0.5))[0][0]

map50_per_class = []
map50_95_per_class = []
precision_per_class = []
recall_per_class = []
f1_per_class = []

for i, class_name in enumerate(class_names):
    if i in class_id_to_idx:
        idx = class_id_to_idx[i]
        ap50 = map_result.ap_per_class[idx, iou50_idx]
        ap50_95 = np.mean(map_result.ap_per_class[idx])
    else:
        ap50, ap50_95 = 0.0, 0.0

    map50_per_class.append(ap50)
    map50_95_per_class.append(ap50_95)

    tp = matrix[i, i]
    fp = np.sum(matrix[:, i]) - tp
    fn = np.sum(matrix[i, :]) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    precision_per_class.append(precision)
    recall_per_class.append(recall)
    f1_per_class.append(f1_score)

results_dict = {
    "Class": class_names,
    "mAP@50": map50_per_class,
    "mAP@50-95": map50_95_per_class,
    "Precision": precision_per_class,
    "Recall": recall_per_class,
    "F1-Score": f1_per_class
}

df_results = pd.DataFrame(results_dict)

print("\n" + "="*75)
print(f"CLASS-WISE METRICS (Confidence Threshold > {CONFIDENCE_THRESHOLD})")
print("="*75)
print(df_results.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

print("\n" + "="*75)
print("GLOBAL METRICS")
print("="*75)
print(f"mAP@50:    {map_result.map50:.4f}")
print(f"mAP@50-95: {map_result.map50_95:.4f}")
print(f"Precision: {np.mean(precision_per_class):.4f}")
print(f"Recall:    {np.mean(recall_per_class):.4f}")
print(f"F1-Score:  {np.mean(f1_per_class):.4f}")